# Customer Purchase Prediction

A PyTorch neural network predicting purchase likelihood from customer browsing session data (synthetic e-commerce dataset).

**Data description**

| Column | Type | Notes |
|---|---|---|
| customer_id | Integer | Unique identifier |
| time_spent | Float | Minutes spent on site per session |
| pages_viewed | Integer | Number of pages viewed |
| basket_value | Float | Value of items in basket |
| device_type | String | Mobile / Desktop / Tablet / Unknown |
| customer_type | String | New / Returning |
| purchase | Binary | Target variable: purchase made (1) or not (0) |


## Step 1: Clean the raw data

Handle missing values with column-appropriate strategies.

In [ ]:
import pandas as pd

clean_data = pd.read_csv('data/raw_customer_data.csv')

clean_data['customer_id'] = clean_data['customer_id'].astype(int)

clean_data['time_spent'] = clean_data['time_spent'].astype(float)
clean_data['time_spent'] = clean_data['time_spent'].fillna(clean_data['time_spent'].median())

clean_data['pages_viewed'] = clean_data['pages_viewed'].fillna(clean_data['pages_viewed'].mean())
clean_data['pages_viewed'] = clean_data['pages_viewed'].astype(int)

clean_data['basket_value'] = clean_data['basket_value'].astype(float)
clean_data['basket_value'] = clean_data['basket_value'].fillna(0)

clean_data['device_type'] = clean_data['device_type'].fillna('Unknown')
clean_data['customer_type'] = clean_data['customer_type'].fillna('New')

clean_data['purchase'] = clean_data['purchase'].astype(int)

clean_data

### Check class balance

Before modeling, check whether `purchase` is balanced — this determines whether accuracy alone will be a meaningful metric.

In [ ]:
print(clean_data['purchase'].value_counts())
print(clean_data['purchase'].value_counts(normalize=True))

The target is imbalanced (roughly 80/20 in favor of `purchase = 1`). A model that always predicts 1 would already score ~80% accuracy without learning anything useful. This means:
- Accuracy alone is a misleading metric here.
- The loss function should account for the imbalance so the model isn't incentivized to just predict the majority class.
- We need to report precision/recall/F1 per class to see whether the model is actually distinguishing buyers from non-buyers.

## Step 2: Prepare features for modeling

Scale numerical features to 0-1 and one-hot encode categoricals.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

model_data = pd.read_csv('data/model_data.csv')
model_feature_set = model_data.copy()

num_cols = ['time_spent', 'pages_viewed', 'basket_value']
scaler = MinMaxScaler()
model_feature_set[num_cols] = scaler.fit_transform(model_feature_set[num_cols])

model_feature_set = pd.get_dummies(model_feature_set, columns=['device_type', 'customer_type'])

dummy_cols = [c for c in model_feature_set.columns
              if c.startswith('device_type_') or c.startswith('customer_type_')]
model_feature_set[dummy_cols] = model_feature_set[dummy_cols].astype(int)

model_feature_set

## Step 3: Build, train, and evaluate the network

A small feedforward network (1 hidden layer, 8 units, ReLU → sigmoid). To address the class imbalance found in Step 1, we weight the loss function so the minority class (`purchase = 0`) isn't drowned out during training.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

torch.manual_seed(42)

train_df = pd.read_csv('data/input_model_features.csv')
val_df = pd.read_csv('data/validation_features.csv')

feature_cols = [c for c in train_df.columns if c not in ('customer_id', 'purchase')]

X = train_df[feature_cols].values.astype(np.float32)
y = train_df['purchase'].values.astype(np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_t = torch.tensor(X_train)
y_train_t = torch.tensor(y_train)
X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

# Class weight for the positive class, to correct for imbalance.
# BCEWithLogitsLoss's pos_weight up-weights the *positive* class; since our
# imbalance runs the other way (purchase=1 is the majority), we instead
# scale the *negative* class by using pos_weight = (n_pos / n_neg) inverted
# via sample weighting below for clarity.
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight_value = n_neg / n_pos  # down-weights the already-dominant positive class
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32)

print(f"Training set purchase=1 proportion: {n_pos / len(y_train):.3f}")
print(f"pos_weight used in loss: {pos_weight_value:.3f}")

class PurchaseNet(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.hidden = nn.Linear(input_dim, 8)
        self.relu = nn.ReLU()
        self.output = nn.Linear(8, 1)

    def forward(self, x):
        x = self.relu(self.hidden(x))
        x = self.output(x)  # raw logits; sigmoid applied via loss/inference
        return x

purchase_model = PurchaseNet(input_dim=X_train.shape[1])

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(purchase_model.parameters(), lr=0.01)

epochs = 200
for epoch in range(epochs):
    purchase_model.train()
    optimizer.zero_grad()
    outputs = purchase_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

purchase_model.eval()
with torch.no_grad():
    test_logits = purchase_model(X_test_t)
    test_probs = torch.sigmoid(test_logits)
    test_preds = (test_probs >= 0.5).float()
    accuracy = (test_preds.eq(y_test_t).sum() / y_test_t.shape[0]).item()

print(f"\nHeld-out validation accuracy: {accuracy:.4f}\n")
print("Classification report (held-out split):")
print(classification_report(y_test_t.numpy(), test_preds.numpy(), target_names=['no purchase', 'purchase']))
print("Confusion matrix:")
print(confusion_matrix(y_test_t.numpy(), test_preds.numpy()))

## Step 4: Predict on the validation set

In [ ]:
X_val = val_df[feature_cols].values.astype(np.float32)
X_val_t = torch.tensor(X_val)

with torch.no_grad():
    val_logits = purchase_model(X_val_t)
    val_probs = torch.sigmoid(val_logits)
    val_preds = (val_probs >= 0.5).int().numpy().flatten()

validation_predictions = pd.DataFrame({
    'customer_id': val_df['customer_id'],
    'purchase': val_preds
})

print("Predicted purchase=1 count:", validation_predictions['purchase'].sum(),
      "out of", len(validation_predictions))
validation_predictions

## What I'd improve next
- Try k-fold cross-validation instead of a single train/test split for a more robust estimate
- Compare against a simple logistic regression baseline to check whether the neural net earns its added complexity
- Tune the classification threshold (not just 0.5) based on the marketing team's actual cost tradeoff between false positives and false negatives
- Try oversampling (e.g. SMOTE) as an alternative to loss weighting, and compare results

## Note
Dataset is synthetic, created for a training exercise — not real customer data.